<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_cell_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Cell-Level Hybrid Engineering + ML Telecom Power Prediction

## Google Colab Python Code

# ============================================================
# CELL-LEVEL HYBRID ENGINEERING + ML MODEL
# TELECOM SITE POWER PREDICTION
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor

# ============================================================
# LOAD FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)
site_db.head(2)

,#,Site_ID,Site Name,2G RRUs,3G RRUs,4G RRUs,5G AAUs,2G Boards,3G Boards,4G Boards,5G Boards,BBU 5900,BBU 3900,BBU 3910
0,1,101,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,2,102,AIRPORT_PRASLIN,2,4,4,0,1,1,1,0,0,1,0


In [9]:
# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [
    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'
]

In [10]:
# ============================================================
# MERGE SITE INFO TO 4G DATA
# ============================================================

lte_df = traffic_4g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

In [11]:
# ============================================================
# MERGE SITE INFO TO 5G DATA
# ============================================================

nr_df = traffic_5g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

In [12]:
# ============================================================
# FILL NULLS
# ============================================================

lte_df.fillna(0, inplace=True)
nr_df.fillna(0, inplace=True)

In [13]:
# ============================================================
# ENGINEERING POWER ASSUMPTIONS
# ============================================================

RRU_2G_POWER = 150
RRU_3G_POWER = 200
RRU_4G_POWER = 180
AAU_5G_POWER = 500

BBU3900_POWER = 55
BBU3910_POWER = 65
BBU5900_POWER = 75

BOARD_4G_POWER = 42.5
BOARD_5G_POWER = 80

In [14]:
# ============================================================
# LTE RRU COUNT PER SITE
# ============================================================

lte_rru_count = (

    lte_df.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()

)

lte_df['lte_rru_count'] = (
    lte_df['Site_ID'].map(lte_rru_count)
)

In [15]:
# ============================================================
# NR AAU COUNT PER SITE
# ============================================================

nr_aau_count = (

    nr_df.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()

)

nr_df['nr_aau_count'] = (
    nr_df['Site_ID'].map(nr_aau_count)
)

In [16]:
# ============================================================
# LTE TRAFFIC BANDS
# ============================================================

lte_df['lte_traffic_max'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [15, 30, 45, 60, 75, 100],

    default=125
)

In [17]:
# ============================================================
# LTE POWER BANDS
# ============================================================

lte_df['bbu_power_band'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [2, 4, 6, 8, 10, 12],

    default=14
)

lte_df['board_power_band'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [3, 6, 9, 12, 15, 18],

    default=21
)

lte_df['rru_power_band'] = np.select(

    [lte_df['traffic_load_mbps'] < 15,
     lte_df['traffic_load_mbps'] < 30,
     lte_df['traffic_load_mbps'] < 45,
     lte_df['traffic_load_mbps'] < 60,
     lte_df['traffic_load_mbps'] < 75,
     lte_df['traffic_load_mbps'] < 100],

    [10, 20, 30, 40, 50, 55],

    default=60
)

In [18]:
# ============================================================
# LTE ENGINEERING POWER
# ============================================================

lte_df['bbu3900_bp'] = np.ceil(((lte_df['BBU3900'] * BBU3900_POWER) / lte_df['lte_rru_count']) * 100) / 100

lte_df['bbu3910_bp'] = np.ceil(((lte_df['BBU3910'] * BBU3910_POWER) / lte_df['lte_rru_count']) * 100) / 100

lte_df['board_4g_bp'] = np.ceil(((lte_df['Boards_4G'] * BOARD_4G_POWER) / lte_df['lte_rru_count']) * 100) / 100

lte_df['rru_4g_bp'] = RRU_4G_POWER

lte_df['bbu_extra_power'] = np.ceil(((((lte_df['traffic_load_mbps'] / lte_df['lte_traffic_max']) * lte_df['bbu_power_band']) / lte_df['lte_rru_count']) * 100)) / 100

lte_df['board_extra_power'] = np.ceil(((((lte_df['traffic_load_mbps'] / lte_df['lte_traffic_max']) * lte_df['board_power_band']) / lte_df['lte_rru_count']) * 100)) / 100

lte_df['rru_extra_power'] = np.ceil((((lte_df['traffic_load_mbps'] / lte_df['lte_traffic_max']) * lte_df['rru_power_band']) * 100)) / 100

lte_df['calc_lte_sec_power'] = (

    lte_df['bbu3900_bp'] +
    lte_df['bbu3910_bp'] +
    lte_df['board_4g_bp'] +
    lte_df['rru_4g_bp'] +
    lte_df['bbu_extra_power'] +
    lte_df['board_extra_power'] +
    lte_df['rru_extra_power']

)

In [20]:
# ============================================================
# NR TRAFFIC BANDS
# ============================================================

nr_df['max_5g_traffic'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [60, 200, 400, 800, 1200, 1300],

    default=1300
)

In [21]:
# ============================================================
# NR POWER BANDS
# ============================================================

nr_df['bbu_power_band'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [2, 4, 6, 8, 10, 12],

    default=14
)

nr_df['board_power_band'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [5, 10, 15, 20, 25, 30],

    default=35
)

nr_df['aau_power_band'] = np.select(

    [nr_df['traffic_load_mbps'] < 60,
     nr_df['traffic_load_mbps'] < 200,
     nr_df['traffic_load_mbps'] < 400,
     nr_df['traffic_load_mbps'] < 800,
     nr_df['traffic_load_mbps'] < 1200,
     nr_df['traffic_load_mbps'] < 1300],

    [50, 100, 150, 200, 250, 300],

    default=350
)

In [22]:
# ============================================================
# NR ENGINEERING POWER
# ============================================================

nr_df['bbu5900_bp'] = np.ceil(((nr_df['BBU5900'] * BBU5900_POWER) / nr_df['nr_aau_count']) * 100) / 100

nr_df['board_5g_bp'] = np.ceil(((nr_df['Boards_5G'] * BOARD_5G_POWER) / nr_df['nr_aau_count']) * 100) / 100

nr_df['aau_5g_bp'] = AAU_5G_POWER

nr_df['bbu_extra_power'] = np.ceil(((((nr_df['traffic_load_mbps'] / nr_df['max_5g_traffic']) * nr_df['bbu_power_band']) / nr_df['nr_aau_count']) * 100)) / 100

nr_df['board_extra_power'] = np.ceil(((((nr_df['traffic_load_mbps'] / nr_df['max_5g_traffic']) * nr_df['board_power_band']) / nr_df['nr_aau_count']) * 100)) / 100

nr_df['aau_extra_power'] = np.ceil((((nr_df['traffic_load_mbps'] / nr_df['max_5g_traffic']) * nr_df['aau_power_band']) * 100)) / 100

nr_df['calc_5g_sec_power'] = (

    nr_df['bbu5900_bp'] +
    nr_df['board_5g_bp'] +
    nr_df['aau_5g_bp'] +
    nr_df['bbu_extra_power'] +
    nr_df['board_extra_power'] +
    nr_df['aau_extra_power']

)

In [24]:
# ============================================================
# LTE SITE POWER AGGREGATION
# ============================================================

lte_site_power = (

    lte_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['calc_lte_sec_power']

    .sum()

)

In [25]:
# ============================================================
# NR SITE POWER AGGREGATION
# ============================================================

nr_site_power = (

    nr_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['calc_5g_sec_power']

    .sum()

)

In [26]:
# ============================================================
# MERGE LTE + NR SITE POWER
# ============================================================

site_prediction = lte_site_power.merge(

    nr_site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='outer'

)

site_prediction.fillna(0, inplace=True)

In [ ]:
# ============================================================
# MERGE ACTUAL SITE POWER
# ============================================================

site_prediction = site_prediction.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

site_prediction.fillna(0, inplace=True)

In [ ]:
# ============================================================
# ADD 2G + 3G POWER
# ============================================================

site_prediction = site_prediction.merge(

    site_db[['Site_ID', 'RRU_2G', 'RRU_3G']],
    on='Site_ID',
    how='left'

)

site_prediction['power_2g'] = (
    site_prediction['RRU_2G'] * RRU_2G_POWER
)

site_prediction['power_3g'] = (
    site_prediction['RRU_3G'] * RRU_3G_POWER
)

In [ ]:
# ============================================================
# FINAL ENGINEERING PREDICTION
# ============================================================

site_prediction['engineering_predicted_power'] = (

    site_prediction['power_2g'] +
    site_prediction['power_3g'] +
    site_prediction['calc_lte_sec_power'] +
    site_prediction['calc_5g_sec_power']

)

In [ ]:
# ============================================================
# RESIDUAL ERROR
# ============================================================

site_prediction['residual_error'] = (

    site_prediction['site_power'] -
    site_prediction['engineering_predicted_power']

)

In [ ]:
# ============================================================
# MACHINE LEARNING FEATURES
# ============================================================

features = [

    'power_2g',
    'power_3g',
    'calc_lte_sec_power',
    'calc_5g_sec_power',
    'engineering_predicted_power'

]

X = site_prediction[features]

y = site_prediction['residual_error']

In [ ]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)

In [ ]:
# ============================================================
# RANDOM FOREST MODEL
# ============================================================

rf_model = RandomForestRegressor(

    n_estimators=100,
    random_state=42

)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

In [ ]:
# ============================================================
# FINAL PREDICTIONS
# ============================================================

engineering_test = site_prediction.loc[
    X_test.index,
    'engineering_predicted_power'
]

final_predictions = (
    engineering_test + rf_predictions
)

actual_values = site_prediction.loc[
    X_test.index,
    'site_power'
]

In [ ]:
# ============================================================
# MODEL EVALUATION
# ============================================================

mae = mean_absolute_error(
    actual_values,
    final_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        actual_values,
        final_predictions
    )
)

mape = np.mean(

    np.abs(
        (actual_values - final_predictions)
        /
        actual_values
    )

) * 100

r2 = r2_score(
    actual_values,
    final_predictions
)

In [ ]:
# ============================================================
# PRINT RESULTS
# ============================================================

print('================================')
print('MODEL PERFORMANCE')
print('================================')

print(f'MAE  : {round(mae, 2)}')
print(f'RMSE : {round(rmse, 2)}')
print(f'MAPE : {round(mape, 2)} %')
print(f'R2   : {round(r2, 4)}')

In [ ]:
# ============================================================
# FINAL OUTPUT TABLE
# ============================================================

results_df = site_prediction.loc[

    X_test.index,
    [
        'Site_ID',
        'trigger_ID',
        'date',
        'datetime',
        'site_power',
        'engineering_predicted_power'
    ]

].copy()

results_df['ml_correction'] = rf_predictions

results_df['final_predicted_power'] = final_predictions

results_df['error'] = (

    results_df['site_power'] -
    results_df['final_predicted_power']

)

results_df['error_percentage'] = (

    np.abs(results_df['error'])
    /
    results_df['site_power']

) * 100

In [23]:
# ============================================================
# EXPORT RESULTS
# ============================================================

results_df.to_excel(
    'Final_Cell_Level_Hybrid_Predictions.xlsx',
    index=False
)

print('================================')
print('OUTPUT FILE CREATED')
print('================================')

print('Final_Cell_Level_Hybrid_Predictions.xlsx')

# ============================================================
# SAMPLE RESULTS
# ============================================================

print(results_df.head(20))


MODEL PERFORMANCE
MAE  : 152.31
RMSE : 213.14
MAPE : 3.73 %
R2   : 0.9894
OUTPUT FILE CREATED
Final_Cell_Level_Hybrid_Predictions.xlsx
       Site_ID  trigger_ID        date          datetime  site_power  \
45078      173           8  2026-03-06  2026-03-06 01:45   5053.2640   
36590      157          44  2026-03-02  2026-03-02 10:45   2856.6377   
9430       115           4  2026-03-02  2026-03-02 00:45   2189.4048   
7096       111          54  2026-03-06  2026-03-06 13:15   1786.0620   
20418      132          37  2026-03-07  2026-03-07 09:00   1293.2444   
3767       106          59  2026-03-02  2026-03-02 14:30   2938.0088   
14164      122           8  2026-03-04  2026-03-04 01:45   6761.5280   
48071      183          52  2026-03-03  2026-03-03 12:45   5416.7300   
20628      132          67  2026-03-07  2026-03-07 16:30   1348.4200   
508        101          73  2026-03-05  2026-03-05 18:00   6430.3190   
33864      153          38  2026-03-06  2026-03-06 09:15   1753.5570   
2